In [1]:
from pathlib import Path

import duckdb
import pandas as pd

con = duckdb.connect()


def query_df(query):
    return con.sql(query).df()


dataset_root = Path("../dataset")

host_logs = dataset_root / "host_logs.parquet"
ground_truth = dataset_root / "ground_truth.parquet"
syscall_table = dataset_root / "syscall_32.tbl"
syscall_lookup = dataset_root / "syscall_32.parquet"

print(f"dataset_root: {dataset_root}")
print(f"host_logs: {host_logs}")
print(f"ground_truth: {ground_truth}")
print(f"syscall_table: {syscall_table}")
print(f"syscall_lookup: {syscall_lookup}")



dataset_root: ../dataset
host_logs: ../dataset/host_logs.parquet
ground_truth: ../dataset/ground_truth.parquet
syscall_table: ../dataset/syscall_32.tbl
syscall_lookup: ../dataset/syscall_32.parquet


# Analysis

Dataset inspection, label distribution, ground-truth checks, duplicate checks, corrupted-row checks, and exploratory attack analysis.


## Block 1: Dataset Scope

**Analysis Question.** What is the overall scope of the host-log dataset and the anomaly label?


In [2]:
query_df(f"""
SELECT 'host_logs' AS dataset, count(*) AS row_count FROM '{host_logs}'
UNION ALL
SELECT 'ground_truth' AS dataset, count(*) AS row_count FROM '{ground_truth}'
""")


,dataset,row_count
0,host_logs,90054239
1,ground_truth,313926


In [3]:
query_df(f"""
SELECT
    count(*) AS row_count,
    count(*) FILTER (WHERE label = 1) AS anomaly_rows,
    round(100.0 * count(*) FILTER (WHERE label = 1) / count(*), 4) AS anomaly_rate_pct,
    min(date) AS min_date,
    max(date) AS max_date,
    count(DISTINCT path) AS distinct_paths,
    count(DISTINCT pro_id) AS distinct_pro_ids,
    count(DISTINCT sys_call) AS distinct_sys_calls
FROM '{host_logs}'
""")



,row_count,anomaly_rows,anomaly_rate_pct,min_date,max_date,distinct_paths,distinct_pro_ids,distinct_sys_calls
0,90054239,1262427,1.4019,2016-03-11,2016-03-16,100,5576,122


**Short inference.** `host_logs.parquet` has around 90054239 rows, among that 1262427 rows are attack rows. This is simulated for 5 days from 11/03/2016 to 16/03/2016.


In [4]:
query_df(f"DESCRIBE SELECT * FROM '{host_logs}'")


,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,time,TIME,YES,None,None,None
2,pro_id,BIGINT,YES,None,None,None
3,path,VARCHAR,YES,None,None,None
4,sys_call,BIGINT,YES,None,None,None
5,event_id,BIGINT,YES,None,None,None
6,attack_cat,VARCHAR,YES,None,None,None
7,attack_subcat,VARCHAR,YES,None,None,None
8,label,BIGINT,YES,None,None,None


In [5]:
query_df(f"SELECT * FROM '{host_logs}' where label=1 LIMIT 10")


,date,time,pro_id,path,sys_call,event_id,attack_cat,attack_subcat,label
0,2016-03-11,02:48:03,2110,/usr/lib/libreoffice/program/soffice.bin,102,57713,Exploits,Office Document Batch,1
1,2016-03-11,02:48:03,2110,/usr/lib/libreoffice/program/soffice.bin,102,57763,Exploits,Office Document Batch,1
2,2016-03-11,02:48:03,2110,/usr/lib/libreoffice/program/soffice.bin,265,57659,Exploits,Office Document Batch,1
3,2016-03-11,02:48:03,2110,/usr/lib/libreoffice/program/soffice.bin,265,57707,Exploits,Office Document Batch,1
4,2016-03-11,02:48:03,2110,/usr/lib/libreoffice/program/soffice.bin,265,57749,Exploits,Office Document Batch,1
5,2016-03-11,02:48:03,2110,/usr/lib/libreoffice/program/soffice.bin,3,57701,Exploits,Office Document Batch,1
6,2016-03-11,02:48:03,4493,/usr/lib/libreoffice/program/soffice.bin,256,57740,Exploits,Office Document Batch,1
7,2016-03-11,02:48:03,4493,/usr/lib/libreoffice/program/soffice.bin,78,57686,Exploits,Office Document Batch,1
8,2016-03-11,02:48:03,4493,/usr/lib/libreoffice/program/soffice.bin,78,57694,Exploits,Office Document Batch,1
9,2016-03-11,02:48:03,4493,/usr/lib/libreoffice/program/soffice.bin,78,57718,Exploits,Office Document Batch,1


In [6]:
query_df(f"select event_id,count(*) as cnt from '{host_logs}' group by event_id having cnt>1")

,event_id,cnt
0,75179406,2
1,75180307,2
2,75179663,2
3,75180603,2
4,75181100,2
...,...,...
344293,75196629,2
344294,75199481,2
344295,75199296,2
344296,75200586,2


In [7]:
query_df(f"select * from '{host_logs}' where event_id in(64351091)")

,date,time,pro_id,path,sys_call,event_id,attack_cat,attack_subcat,label
0,2016-03-13,23:56:37,2110,/usr/bin/compiz,54,64351091,normal,normal,0
1,2016-03-13,23:56:37,2110,/usr/bin/compiz,54,64351091,normal,normal,0


In [9]:
query_df(f"SELECT * FROM '{host_logs}' where label=1 LIMIT 10")


,date,time,pro_id,path,sys_call,event_id,attack_cat,attack_subcat,label
0,2016-03-11,02:48:03,2110,/usr/lib/libreoffice/program/soffice.bin,102,57713,Exploits,Office Document Batch,1
1,2016-03-11,02:48:03,2110,/usr/lib/libreoffice/program/soffice.bin,102,57763,Exploits,Office Document Batch,1
2,2016-03-11,02:48:03,2110,/usr/lib/libreoffice/program/soffice.bin,265,57659,Exploits,Office Document Batch,1
3,2016-03-11,02:48:03,2110,/usr/lib/libreoffice/program/soffice.bin,265,57707,Exploits,Office Document Batch,1
4,2016-03-11,02:48:03,2110,/usr/lib/libreoffice/program/soffice.bin,265,57749,Exploits,Office Document Batch,1
5,2016-03-11,02:48:03,2110,/usr/lib/libreoffice/program/soffice.bin,3,57701,Exploits,Office Document Batch,1
6,2016-03-11,02:48:03,4493,/usr/lib/libreoffice/program/soffice.bin,256,57740,Exploits,Office Document Batch,1
7,2016-03-11,02:48:03,4493,/usr/lib/libreoffice/program/soffice.bin,78,57686,Exploits,Office Document Batch,1
8,2016-03-11,02:48:03,4493,/usr/lib/libreoffice/program/soffice.bin,78,57694,Exploits,Office Document Batch,1
9,2016-03-11,02:48:03,4493,/usr/lib/libreoffice/program/soffice.bin,78,57718,Exploits,Office Document Batch,1


In [10]:
query_df(f"select * from '{ground_truth}' where attack_cat='Exploits' LIMIT 10")

,date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips
0,2016-03-11,3:07:12,Exploits,Clientside,Oracle Outside In XPM Image Processing Stack O...,CVSS-Critical (https://strikecenter.bpointsys....,TCP 175.45.176.1:13276->10.40.85.32:25
1,2016-03-14,10:33:36,Exploits,Clientside,Oracle Outside In XPM Image Processing Stack O...,CVSS-Critical (https://strikecenter.bpointsys....,TCP 175.45.176.2:19537->10.40.85.32:25
2,2016-03-14,12:43:12,Exploits,Clientside,Oracle Outside In XPM Image Processing Stack O...,CVSS-Critical (https://strikecenter.bpointsys....,TCP 175.45.176.0:7727->10.40.85.32:25
3,2016-03-11,8:10:17,Exploits,Clientside Microsoft Office Batch,Microsoft Office Excel Formula Record PtgExtra...,CVE 2010-3239 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.3:45583->10.40.85.32:25
4,2016-03-11,4:48:33,Exploits,Clientside Microsoft Office Batch,Microsoft Office Excel Formula Record PtgExtra...,CVE 2010-3231 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.1:4714->10.40.85.32:25
5,2016-03-14,9:45:22,Exploits,Clientside Microsoft Office Batch,Microsoft Word ActiveX ScriptBridge Double Fre...,CVE 2010-3331 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.1:27913->10.40.85.32:25
6,2016-03-11,6:08:30,Exploits,Clientside Microsoft Office Batch,Microsoft Office Powerpoint OEPlaceHolderAtom ...,CVE 2010-0032 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.2:13285->10.40.85.32:25
7,2016-03-14,12:14:24,Exploits,Clientside Microsoft Office Batch,Microsoft Office Powerpoint Legacy File Parsin...,CVE 2010-2572 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.2:16441->10.40.85.32:25
8,2016-03-11,7:26:24,Exploits,Clientside Microsoft Office Batch,Microsoft Office Excel Formula Record PtgExtra...,CVE 2010-3239 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.1:2005->10.40.85.32:25
9,2016-03-11,8:09:36,Exploits,Clientside Microsoft Office Batch,Microsoft Office Excel Formula Record PtgExtra...,CVE 2010-3231 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.1:26343->10.40.85.32:25


In [11]:
query_df(f"""
SELECT DISTINCT attack_cat
FROM '{ground_truth}'
WHERE attack_cat NOT LIKE '%->%'
  AND attack_cat NOT LIKE '%:%'
""")

,attack_cat
0,Fuzzers
1,Exploits
2,Malware
3,Exploits
4,Backdoors
5,Generic
6,Reconnaissance
7,Denial of Service
8,Denial of Service
9,Shellcode


In [12]:

query_df(f"select * from '{ground_truth}' where attack_cat like '%->%'")

,date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips
0,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:30961->10.40.85.32:80,3:21:36,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
1,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.1:32954->10.40.85.32:80,10:48:00,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
2,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:30310->10.40.85.32:80,7:26:24,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
3,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:46216->10.40.85.32:80,4:33:36,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
4,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.3:12939->10.40.85.32:80,1:26:24,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
...,...,...,...,...,...,...,...
295,2016-03-11,CVE 2009-1535 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:7000->10.40.85.32:80,2:52:48,Exploits,Microsoft IIS Batch,"Microsoft IIS 5.0, IIS 5.1, IIS 6.0 WebDAV Aut..."
296,2016-03-11,CVE 2009-1535 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.0:51081->10.40.85.32:80,12:43:12,Exploits,Microsoft IIS Batch,"Microsoft IIS 5.0, IIS 5.1, IIS 6.0 WebDAV Aut..."
297,2016-03-11,CVE 2009-1535 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:42736->10.40.85.32:80,12:43:12,Exploits,Microsoft IIS Batch,"Microsoft IIS 5.0, IIS 5.1, IIS 6.0 WebDAV Aut..."
298,2016-03-11,CVE 2009-1535 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.0:17101->10.40.85.32:80,7:12:00,Exploits,Microsoft IIS Batch,"Microsoft IIS 5.0, IIS 5.1, IIS 6.0 WebDAV Aut..."


In [13]:
query_df(f"""
SELECT
    COUNT(*) AS total_rows,
    (
        SELECT COUNT(*)
        FROM '{ground_truth}'
        WHERE attack_cat LIKE '%->%'
          AND attack_cat LIKE '%:%'
    ) AS corrupted_rows
FROM '{ground_truth}'
""")

,total_rows,corrupted_rows
0,313926,300


In [14]:
query_df(f"DESCRIBE '{host_logs}'")


,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,time,TIME,YES,None,None,None
2,pro_id,BIGINT,YES,None,None,None
3,path,VARCHAR,YES,None,None,None
4,sys_call,BIGINT,YES,None,None,None
5,event_id,BIGINT,YES,None,None,None
6,attack_cat,VARCHAR,YES,None,None,None
7,attack_subcat,VARCHAR,YES,None,None,None
8,label,BIGINT,YES,None,None,None


In [84]:
query_df(f"DESCRIBE '{ground_truth}'")

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,time,VARCHAR,YES,None,None,None
2,attack_cat,VARCHAR,YES,None,None,None
3,attack_subcat,VARCHAR,YES,None,None,None
4,attack_name,VARCHAR,YES,None,None,None
5,attack_refrence,VARCHAR,YES,None,None,None
6,ips,VARCHAR,YES,None,None,None


In [15]:
query_df(f"""
SELECT pro_id,count(*) AS cnt
FROM '{host_logs}'
WHERE date = '2016-03-11'
  AND time = '3:07:12'
  and label = 1
GROUP BY pro_id
""")

,pro_id,cnt
0,1081,96
1,4519,40
2,2110,195
3,1853,4
4,4461,2
5,4493,40


In [39]:
query_df(f"select * from '{ground_truth}' where date='2016-03-11' and time='3:07:12' and attack_cat='Exploits'")

,date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips
0,2016-03-11,3:07:12,Exploits,Clientside,Oracle Outside In XPM Image Processing Stack O...,CVSS-Critical (https://strikecenter.bpointsys....,TCP 175.45.176.1:13276->10.40.85.32:25
1,2016-03-11,3:07:12,Exploits,Clientside Microsoft Office Batch,Microsoft Office Excel Formula Record PtgExtra...,CVE 2010-3231 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.2:49215->10.40.85.32:25
2,2016-03-11,3:07:12,Exploits,Clientside Microsoft Office Batch,Microsoft Word ActiveX ScriptBridge Double Fre...,CVE 2010-3331 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.1:30084->10.40.85.32:25
3,2016-03-11,3:07:12,Exploits,Clientside Microsoft Office Batch,Microsoft Office Excel Formula Record PtgExtra...,CVE 2010-3239 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.0:56297->10.40.85.32:25
4,2016-03-11,3:07:12,Exploits,Clientside Microsoft Office Batch,Microsoft Office Powerpoint Legacy File Format...,CVE 2011-0976 (http://cve.mitre.org/cgi-bin/cv...,TCP 175.45.176.0:31203->10.40.85.32:25
...,...,...,...,...,...,...,...
655,2016-03-11,3:07:12,Exploits,Clientside,Microsoft Powerpoint 2003 Heap Overflow (POP3)...,BPS 2011-0001 (https://strikecenter.bpointsys....,175.45.176.0:27210->10.40.85.32:110
656,2016-03-11,3:07:12,Exploits,Clientside,Windows Media Player ASF Media File Format Par...,CVE 2009-2527 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.0:12987->10.40.85.32:110
657,2016-03-11,3:07:12,Exploits,Clientside,Microsoft Office Text Converter Integer Underf...,CVE 2009-0087 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.1:8467->10.40.85.32:143
658,2016-03-11,3:07:12,Exploits,Clientside,Microsoft PowerPoint Viewer TextChars Atom Rec...,CVE 2010-0034 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.0:6803->10.40.85.32:80


In [53]:
query_df(f"select distinct attack_name,attack_cat from '{ground_truth}' where attack_cat!='Exploits'")

,attack_name,attack_cat
0,Digium Asterisk SIP SDP Media Descriptions Con...,Exploits
1,Microsoft Windows SMB MDL Buffer Overflow (htt...,Exploits
2,OnLineGames (https://strikecenter.bpointsys.co...,Malware
3,Backdoor.Win32.Simda.qcc (https://strikecenter...,Malware
4,Dorkbot (https://strikecenter.bpointsys.com/bp...,Malware
...,...,...
2697,Microsoft Office Uninitialized Memory Corrupti...,Exploits
2698,Microsoft Excel XF Record Unchecked Inheritanc...,Exploits
2699,Microsoft WordPad Embedded COM Code Execution ...,Exploits
2700,PHP Batch,2:24:00


In [24]:
query_df(f"SELECT * FROM '{ground_truth}' WHERE  attack_cat='Backdoors' and date='2016-03-11' and time='3:07:12'")

,date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips
0,2016-03-11,3:07:12,Backdoors,All Batch,BlackEnergy Botnet Command and Control Communi...,http://atlas-public.ec2.arbor.net/docs/BlackEn...,175.45.176.3:62151->10.40.85.32:60253
1,2016-03-11,3:07:12,Backdoors,All Batch,Backdoor: Cisco Prime LAN Management (https://...,CVE 2012-6392 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.3:33948->10.40.85.32:514
2,2016-03-11,3:07:12,Backdoors,All Batch,phpmyadmin 3.5.2.2 Backdoor Access and Code Ex...,CVE 2012-5159 (http://cve.mitre.org/cgi-bin/cv...,IP 175.45.176.1:0->10.40.85.32:0 175.45.176.1:...
3,2016-03-11,3:07:12,Backdoors,All Batch,BlackEnergy Botnet Command and Control Communi...,http://atlas-public.ec2.arbor.net/docs/BlackEn...,175.45.176.3:34536->10.40.85.32:56157
4,2016-03-11,3:07:12,Backdoors,All Batch,Symantec LiveUpdate Administrator Security Byp...,CVE 2014-1644 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:44833->10.40.85.32:80
5,2016-03-11,3:07:12,Backdoors,All Batch,Backdoor: Windows 7 CMD.EXE Reverse Shell (htt...,Backdoor: Windows 7 CMD.EXE Reverse Shell (htt...,175.45.176.1:65487->10.40.85.32:60988
6,2016-03-11,3:07:12,Backdoors,All Batch,Vtiger CRM Unauthenticated Password Reset (htt...,CVE 2014-2269 (http://cve.mitre.org/cgi-bin/cv...,IP 175.45.176.0:0->10.40.85.32:0 175.45.176.0:...
7,2016-03-11,3:07:12,Backdoors,All Batch,Zavio IP Camera Firmware 1.6.03 User Credentia...,CVE 2013-2568 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.0:53627->10.40.85.32:80
8,2016-03-11,3:07:12,Backdoors,All Batch,phpmyadmin 3.5.2.2 Backdoor Access and Code Ex...,CVE 2012-5159 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.3:3119->10.40.85.32:80IP 175.45.17...
9,2016-03-11,3:07:12,Backdoors,All Batch,Cisco Network Registrar Default Credentials Ba...,CVE 2011-2024 (http://cve.mitre.org/cgi-bin/cv...,IP 175.45.176.1:0->10.40.85.32:0 175.45.176.1:...


In [23]:
query_df(f"select h.date,h.time,h.path,h.sys_call,h.pro_id,h.attack_cat,s.i386_name from '{host_logs}' h left join '{syscall_lookup}' s on h.sys_call=s.sys_call where date='2016-03-11' and time='3:07:12'")

,date,time,path,sys_call,pro_id,attack_cat,i386_name
0,2016-03-11,03:07:12,/usr/bin/compiz,102,2110,Backdoors,socketcall
1,2016-03-11,03:07:12,/usr/bin/compiz,102,2110,Backdoors,socketcall
2,2016-03-11,03:07:12,/usr/bin/compiz,102,2110,Backdoors,socketcall
3,2016-03-11,03:07:12,/usr/bin/compiz,102,2110,Backdoors,socketcall
4,2016-03-11,03:07:12,/usr/bin/compiz,102,2110,Backdoors,socketcall
...,...,...,...,...,...,...,...
372,2016-03-11,03:07:12,/usr/sbin/apache2,256,4519,Backdoors,epoll_wait
373,2016-03-11,03:07:12,/usr/sbin/apache2,78,4519,Backdoors,gettimeofday
374,2016-03-11,03:07:12,/usr/sbin/apache2,78,4519,Backdoors,gettimeofday
375,2016-03-11,03:07:12,/usr/sbin/apache2,78,4519,Backdoors,gettimeofday


In [99]:
  attack_date = "2016-03-11"
  attack_time = "3:07:12"
  attack_cat = "Backdoors"

  query_df(f"""
  SELECT
      h.date,
      h.time,
      h.pro_id,
      h.path,
      h.sys_call,
      s.i386_name AS syscall_name,
      h.event_id,
      h.attack_cat,
      h.attack_subcat,
      h.label
  FROM '{host_logs}' h
  LEFT JOIN '{syscall_lookup}' s
      ON h.sys_call = s.sys_call
  WHERE h.date = DATE '{attack_date}'
    AND h.time = TIME '{attack_time}'
    AND h.attack_cat = '{attack_cat}'
  ORDER BY h.pro_id, h.path, h.event_id
  LIMIT 100
  """)


,date,time,pro_id,path,sys_call,syscall_name,event_id,attack_cat,attack_subcat,label
0,2016-03-11,03:07:12,1081,/usr/bin/Xorg,265,clock_gettime,356107,Backdoors,All Batch,1
1,2016-03-11,03:07:12,1081,/usr/bin/Xorg,104,setitimer,356108,Backdoors,All Batch,1
2,2016-03-11,03:07:12,1081,/usr/bin/Xorg,265,clock_gettime,356109,Backdoors,All Batch,1
3,2016-03-11,03:07:12,1081,/usr/bin/Xorg,102,socketcall,356110,Backdoors,All Batch,1
4,2016-03-11,03:07:12,1081,/usr/bin/Xorg,146,writev,356111,Backdoors,All Batch,1
...,...,...,...,...,...,...,...,...,...,...
95,2016-03-11,03:07:12,1081,/usr/bin/Xorg,142,_newselect,356499,Backdoors,All Batch,1
96,2016-03-11,03:07:12,1853,/usr/lib/unity-settings-daemon/unity-settings-...,265,clock_gettime,356412,Backdoors,All Batch,1
97,2016-03-11,03:07:12,1853,/usr/lib/unity-settings-daemon/unity-settings-...,102,socketcall,356413,Backdoors,All Batch,1
98,2016-03-11,03:07:12,1853,/usr/lib/unity-settings-daemon/unity-settings-...,265,clock_gettime,356414,Backdoors,All Batch,1


In [31]:
query_df(f"describe '{ground_truth}'")

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,time,VARCHAR,YES,None,None,None
2,attack_cat,VARCHAR,YES,None,None,None
3,attack_subcat,VARCHAR,YES,None,None,None
4,attack_name,VARCHAR,YES,None,None,None
5,attack_refrence,VARCHAR,YES,None,None,None
6,ips,VARCHAR,YES,None,None,None


In [98]:
query_df(f"select date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips,count(*) from '{ground_truth}' group by date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips having count(*)>1")

,date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips,count_star()


There are totally 1324 duplicate rows.

In [41]:
query_df(f"select count(*) from '{ground_truth}'")

,count_star()
0,313926


In [42]:
query_df(f"select count(*) from '{host_logs}' where label=1")

,count_star()
0,1262427


In [50]:
  shifted_df = query_df(f"""
  SELECT
      *
  FROM '{ground_truth}'
  WHERE NOT regexp_matches(trim(time), '^([0-9]|1[0-9]|2[0-3]):[0-5][0-9]:[0-5][0-9]$')
     OR NOT regexp_matches(trim(attack_cat), '^(Exploits|Malware|Denial of Service|Generic|Shellcode|Reconnaissance|Worms|Backdoors|Fuzzers)$')
     OR NOT regexp_matches(trim(ips), '^[0-9]{{1,3}}(\\.[0-9]{{1,3}}){{3}}:[0-9]{{1,5}}->[0-9]{{1,3}}(\\.[0-9]{{1,3}}){{3}}:[0-9]{{1,5}}$')
  """)


In [52]:
  query_df(f"""
  SELECT
      count(*) AS total_rows,
      count(*) FILTER (
          WHERE NOT regexp_full_match(trim(time), '([0-9]|1[0-9]|2[0-3]):[0-5][0-9]:[0-5][0-9]')
      ) AS invalid_time_rows,
      count(*) FILTER (
          WHERE NOT regexp_full_match(trim(attack_cat), '(Exploits|Malware|Denial of Service|Generic|Shellcode|Reconnaissance|Worms|Backdoors|Fuzzers)')
      ) AS invalid_attack_cat_rows,
      count(*) FILTER (
          WHERE NOT regexp_full_match(trim(ips), '[0-9]{{1,3}}(\\.[0-9]{{1,3}}){{3}}:[0-9]{{1,5}}->[0-9]{{1,3}}(\\.[0-9]{{1,3}}){{3}}:[0-9]{{1,5}}')
      ) AS invalid_ips_rows
  FROM '{ground_truth}'
  """)

,total_rows,invalid_time_rows,invalid_attack_cat_rows,invalid_ips_rows
0,313926,527,501,83527


In [54]:
  query_df(f"""
  SELECT
      date,
      time,
      attack_cat,
      attack_subcat,
      attack_name,
      attack_refrence,
      ips
  FROM '{ground_truth}'
  WHERE NOT regexp_full_match(
      trim(time),
      '([0-9]|1[0-9]|2[0-3]):[0-5][0-9]:[0-5][0-9]'
  )
  LIMIT 50
  """)



,date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips
0,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:30961->10.40.85.32:80,3:21:36,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
1,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.1:32954->10.40.85.32:80,10:48:00,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
2,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:30310->10.40.85.32:80,7:26:24,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
3,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:46216->10.40.85.32:80,4:33:36,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
4,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.3:12939->10.40.85.32:80,1:26:24,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
5,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.0:31702->10.40.85.32:80,9:07:12,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
6,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.0:26297->10.40.85.32:80,4:33:36,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
7,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:47193->10.40.85.32:80,2:24:00,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
8,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.1:47961->10.40.85.32:80,8:52:48,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
9,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:44785->10.40.85.32:80,9:50:24,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."


In [64]:
  query_df(f"""
  SELECT
      date,
      time,
      attack_cat,
      attack_subcat,
      attack_name,
      attack_refrence,
      ips
  FROM '{ground_truth}'
  WHERE NOT regexp_full_match(
      trim(attack_cat),
      '(Exploits|Malware|Denial of Service|Generic|Shellcode|Reconnaissance|Worms|Backdoors|Fuzzers)'
  )
  """)


,date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips
0,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:30961->10.40.85.32:80,3:21:36,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
1,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.1:32954->10.40.85.32:80,10:48:00,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
2,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:30310->10.40.85.32:80,7:26:24,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
3,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:46216->10.40.85.32:80,4:33:36,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
4,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.3:12939->10.40.85.32:80,1:26:24,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
...,...,...,...,...,...,...,...
496,2016-03-11,175.45.176.1:59278->10.40.85.32:80,2:24:00,Exploits,Web Application,"Movable Type 4.2x, 4.3x Upgrade Script RCE (ht...",CVE 2012-6315 (http://cve.mitre.org/cgi-bin/cv...
497,2016-03-11,175.45.176.3:64023->10.40.85.32:80,6:43:12,Exploits,Web Application,"Movable Type 4.2x, 4.3x Upgrade Script RCE (ht...",CVE 2012-6315 (http://cve.mitre.org/cgi-bin/cv...
498,2016-03-11,175.45.176.2:18099->10.40.85.32:80,10:33:36,Exploits,Web Application,"Movable Type 4.2x, 4.3x Upgrade Script RCE (ht...",CVE 2012-6315 (http://cve.mitre.org/cgi-bin/cv...
499,2016-03-11,175.45.176.0:8061->10.40.85.32:80,7:12:00,Exploits,Web Application,"Movable Type 4.2x, 4.3x Upgrade Script RCE (ht...",CVE 2012-6315 (http://cve.mitre.org/cgi-bin/cv...


In [59]:
  query_df(f"""
  SELECT
      count(*) AS total_rows,
      count(*) FILTER (
          WHERE NOT regexp_full_match(trim(time), '([0-9]|1[0-9]|2[0-3]):[0-5][0-9]:[0-5][0-9]')
      ) AS invalid_time_rows,
      count(*) FILTER (
          WHERE NOT regexp_full_match(trim(attack_cat), '(Exploits|Malware|Denial of Service|Generic|Shellcode|Reconnaissance|Worms|Backdoors|Fuzzers)')
      ) AS invalid_attack_cat_rows,
      count(*) FILTER (
          WHERE NOT regexp_full_match(trim(ips), '[0-9]{{1,3}}(\\.[0-9]{{1,3}}){{3}}:[0-9]{{1,5}}->[0-9]{{1,3}}(\\.[0-9]{{1,3}}){{3}}:[0-9]{{1,5}}')
      ) AS invalid_ips_rows
  FROM '{ground_truth}'
  """)


,total_rows,invalid_time_rows,invalid_attack_cat_rows,invalid_ips_rows
0,313926,527,501,83527


In [66]:
query_df(f"select count(*) from '{ground_truth}' where time='Time'")

,count_star()
0,26


#We have 501 corrupted rows in the ground truth, lets recover it.

,date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips
0,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:30961->10.40.85.32:80,3:21:36,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
1,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.1:32954->10.40.85.32:80,10:48:00,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
2,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:30310->10.40.85.32:80,7:26:24,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
3,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:46216->10.40.85.32:80,4:33:36,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
4,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.3:12939->10.40.85.32:80,1:26:24,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
5,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.0:31702->10.40.85.32:80,9:07:12,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
6,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.0:26297->10.40.85.32:80,4:33:36,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
7,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:47193->10.40.85.32:80,2:24:00,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
8,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.1:47961->10.40.85.32:80,8:52:48,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
9,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:44785->10.40.85.32:80,9:50:24,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."


In [62]:
  query_df(f"""
  SELECT
      date,
      time,
      attack_cat,
      attack_subcat,
      attack_name,
      attack_refrence,
      ips
  FROM '{ground_truth}'
  WHERE regexp_full_match(
      trim(time),
      '[0-9]{{1,3}}(\\.[0-9]{{1,3}}){{3}}:[0-9]{{1,5}}->[0-9]{{1,3}}(\\.[0-9]{{1,3}}){{3}}:[0-9]{{1,5}}'
  )
  LIMIT 20
  """)


,date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips
0,2016-03-11,175.45.176.1:45992->10.40.85.32:80,4:48:00,Denial of Service,Browser Batch,"1,686.02",CVE 2012-1874 (http://cve.mitre.org/cgi-bin/cv...
1,2016-03-11,175.45.176.0:24161->10.40.85.32:80,11:02:24,Exploits,PHP Batch,"PHP Charts 1.0 (index.php, type parameter) Rem...",OSVDB 93563 (http://www.osvdb.org/93563)CVSS-C...
2,2016-03-11,175.45.176.3:56959->10.40.85.32:80,3:50:24,Exploits,PHP Batch,"PHP Charts 1.0 (index.php, type parameter) Rem...",OSVDB 93563 (http://www.osvdb.org/93563)CVSS-C...
3,2016-03-11,175.45.176.3:23086->10.40.85.32:80,5:02:24,Exploits,PHP Batch,"PHP Charts 1.0 (index.php, type parameter) Rem...",OSVDB 93563 (http://www.osvdb.org/93563)CVSS-C...
4,2016-03-11,175.45.176.3:38965->10.40.85.32:80,4:33:36,Exploits,PHP Batch,"PHP Charts 1.0 (index.php, type parameter) Rem...",OSVDB 93563 (http://www.osvdb.org/93563)CVSS-C...
5,2016-03-11,175.45.176.1:13266->10.40.85.32:80,8:52:48,Exploits,PHP Batch,"PHP Charts 1.0 (index.php, type parameter) Rem...",OSVDB 93563 (http://www.osvdb.org/93563)CVSS-C...
6,2016-03-11,175.45.176.2:37505->10.40.85.32:80,9:21:36,Exploits,PHP Batch,"PHP Charts 1.0 (index.php, type parameter) Rem...",OSVDB 93563 (http://www.osvdb.org/93563)CVSS-C...
7,2016-03-11,175.45.176.0:61626->10.40.85.32:80,5:45:36,Exploits,PHP Batch,"PHP Charts 1.0 (index.php, type parameter) Rem...",OSVDB 93563 (http://www.osvdb.org/93563)CVSS-C...
8,2016-03-11,175.45.176.1:57699->10.40.85.32:80,11:16:48,Exploits,PHP Batch,"PHP Charts 1.0 (index.php, type parameter) Rem...",OSVDB 93563 (http://www.osvdb.org/93563)CVSS-C...
9,2016-03-11,175.45.176.2:35143->10.40.85.32:80,12:28:48,Exploits,PHP Batch,"PHP Charts 1.0 (index.php, type parameter) Rem...",OSVDB 93563 (http://www.osvdb.org/93563)CVSS-C...


In [69]:
  query_df(f"""
  SELECT *
  FROM '{ground_truth}'
  WHERE NOT regexp_full_match(
      trim(time),
      '([01]?[0-9]|2[0-3]):[0-5][0-9]:[0-5][0-9]'
  )
  AND NOT regexp_full_match(
      trim(attack_cat),
      '(Exploits|Malware|Denial of Service|Generic|Shellcode|Reconnaissance|Worms|Backdoors|Fuzzers)'
  )
  AND NOT regexp_full_match(
      trim(ips),
      '[0-9]{{1,3}}(\\.[0-9]{{1,3}}){{3}}:[0-9]{{1,5}}->[0-9]{{1,3}}(\\.[0-9]{{1,3}}){{3}}:[0-9]{{1,5}}'
  )
  """)


,date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips
0,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:30961->10.40.85.32:80,3:21:36,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
1,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.1:32954->10.40.85.32:80,10:48:00,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
2,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:30310->10.40.85.32:80,7:26:24,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
3,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:46216->10.40.85.32:80,4:33:36,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
4,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.3:12939->10.40.85.32:80,1:26:24,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
...,...,...,...,...,...,...,...
496,2016-03-11,175.45.176.1:59278->10.40.85.32:80,2:24:00,Exploits,Web Application,"Movable Type 4.2x, 4.3x Upgrade Script RCE (ht...",CVE 2012-6315 (http://cve.mitre.org/cgi-bin/cv...
497,2016-03-11,175.45.176.3:64023->10.40.85.32:80,6:43:12,Exploits,Web Application,"Movable Type 4.2x, 4.3x Upgrade Script RCE (ht...",CVE 2012-6315 (http://cve.mitre.org/cgi-bin/cv...
498,2016-03-11,175.45.176.2:18099->10.40.85.32:80,10:33:36,Exploits,Web Application,"Movable Type 4.2x, 4.3x Upgrade Script RCE (ht...",CVE 2012-6315 (http://cve.mitre.org/cgi-bin/cv...
499,2016-03-11,175.45.176.0:8061->10.40.85.32:80,7:12:00,Exploits,Web Application,"Movable Type 4.2x, 4.3x Upgrade Script RCE (ht...",CVE 2012-6315 (http://cve.mitre.org/cgi-bin/cv...


Now lets match with the first patter:

date -> date
time -> attack_reference
attack_cat -> ips
attack_subcat -> time
attack_name -> attack_subcat
attack_reference -> Browser Batch
ips ->

In [71]:
##Check for pattern matching based on time regex

query_df(f"""
  SELECT *
  FROM '{ground_truth}'
  WHERE regexp_full_match(
      trim(attack_cat),
      '[0-9]{{1,3}}(\\.[0-9]{{1,3}}){{3}}:[0-9]{{1,5}}->[0-9]{{1,3}}(\\.[0-9]{{1,3}}){{3}}:[0-9]{{1,5}}'
  )
  """)


,date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips
0,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:30961->10.40.85.32:80,3:21:36,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
1,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.1:32954->10.40.85.32:80,10:48:00,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
2,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:30310->10.40.85.32:80,7:26:24,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
3,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:46216->10.40.85.32:80,4:33:36,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
4,2016-03-11,CVE 2012-0501 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.3:12939->10.40.85.32:80,1:26:24,Denial of Service,Browser Batch,"Oracle Java 5,6,7 ZipFile readCEN Denial of Se..."
...,...,...,...,...,...,...,...
295,2016-03-11,CVE 2009-1535 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:7000->10.40.85.32:80,2:52:48,Exploits,Microsoft IIS Batch,"Microsoft IIS 5.0, IIS 5.1, IIS 6.0 WebDAV Aut..."
296,2016-03-11,CVE 2009-1535 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.0:51081->10.40.85.32:80,12:43:12,Exploits,Microsoft IIS Batch,"Microsoft IIS 5.0, IIS 5.1, IIS 6.0 WebDAV Aut..."
297,2016-03-11,CVE 2009-1535 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:42736->10.40.85.32:80,12:43:12,Exploits,Microsoft IIS Batch,"Microsoft IIS 5.0, IIS 5.1, IIS 6.0 WebDAV Aut..."
298,2016-03-11,CVE 2009-1535 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.0:17101->10.40.85.32:80,7:12:00,Exploits,Microsoft IIS Batch,"Microsoft IIS 5.0, IIS 5.1, IIS 6.0 WebDAV Aut..."


In [72]:
query_df(f"""
  SELECT *
  FROM '{ground_truth}'
  WHERE regexp_full_match(
      trim(time),
      '[0-9]{{1,3}}(\\.[0-9]{{1,3}}){{3}}:[0-9]{{1,5}}->[0-9]{{1,3}}(\\.[0-9]{{1,3}}){{3}}:[0-9]{{1,5}}'
  )
  """)

,date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips
0,2016-03-11,175.45.176.1:45992->10.40.85.32:80,4:48:00,Denial of Service,Browser Batch,"1,686.02",CVE 2012-1874 (http://cve.mitre.org/cgi-bin/cv...
1,2016-03-11,175.45.176.0:24161->10.40.85.32:80,11:02:24,Exploits,PHP Batch,"PHP Charts 1.0 (index.php, type parameter) Rem...",OSVDB 93563 (http://www.osvdb.org/93563)CVSS-C...
2,2016-03-11,175.45.176.3:56959->10.40.85.32:80,3:50:24,Exploits,PHP Batch,"PHP Charts 1.0 (index.php, type parameter) Rem...",OSVDB 93563 (http://www.osvdb.org/93563)CVSS-C...
3,2016-03-11,175.45.176.3:23086->10.40.85.32:80,5:02:24,Exploits,PHP Batch,"PHP Charts 1.0 (index.php, type parameter) Rem...",OSVDB 93563 (http://www.osvdb.org/93563)CVSS-C...
4,2016-03-11,175.45.176.3:38965->10.40.85.32:80,4:33:36,Exploits,PHP Batch,"PHP Charts 1.0 (index.php, type parameter) Rem...",OSVDB 93563 (http://www.osvdb.org/93563)CVSS-C...
...,...,...,...,...,...,...,...
196,2016-03-11,175.45.176.1:59278->10.40.85.32:80,2:24:00,Exploits,Web Application,"Movable Type 4.2x, 4.3x Upgrade Script RCE (ht...",CVE 2012-6315 (http://cve.mitre.org/cgi-bin/cv...
197,2016-03-11,175.45.176.3:64023->10.40.85.32:80,6:43:12,Exploits,Web Application,"Movable Type 4.2x, 4.3x Upgrade Script RCE (ht...",CVE 2012-6315 (http://cve.mitre.org/cgi-bin/cv...
198,2016-03-11,175.45.176.2:18099->10.40.85.32:80,10:33:36,Exploits,Web Application,"Movable Type 4.2x, 4.3x Upgrade Script RCE (ht...",CVE 2012-6315 (http://cve.mitre.org/cgi-bin/cv...
199,2016-03-11,175.45.176.0:8061->10.40.85.32:80,7:12:00,Exploits,Web Application,"Movable Type 4.2x, 4.3x Upgrade Script RCE (ht...",CVE 2012-6315 (http://cve.mitre.org/cgi-bin/cv...


In [8]:
temp_ground_truth = dataset_root / "temp_ground_truth.parquet"
con.execute(f"""
COPY (
    SELECT *
    FROM '{ground_truth}'
    WHERE time != 'Time'
) TO '{temp_ground_truth}'
""")
temp_ground_truth.replace(ground_truth)

PosixPath('../dataset/ground_truth.parquet')

In [9]:
query_df(f"""
  SELECT
      count(*) AS total_rows,
      count(*) FILTER (WHERE time = 'Time') AS header_rows
  FROM '{ground_truth}'
  """)

,total_rows,header_rows
0,313900,0


In [10]:
  ip_tuple_regex = r'[0-9]{1,3}(\.[0-9]{1,3}){3}:[0-9]{1,5}->[0-9]{1,3}(\.[0-9]{1,3}){3}:[0-9]{1,5}'

  fixed_ground_truth = dataset_root / "ground_truth_fixed_tmp.parquet"

  con.execute(f"""
  COPY (
      SELECT
          date,

          CASE
              WHEN regexp_full_match(trim(attack_cat), '{ip_tuple_regex}') THEN trim(attack_subcat)
              WHEN regexp_full_match(trim(time), '{ip_tuple_regex}') THEN trim(attack_cat)
              ELSE trim(time)
          END AS time,

          CASE
              WHEN regexp_full_match(trim(attack_cat), '{ip_tuple_regex}') THEN trim(attack_name)
              WHEN regexp_full_match(trim(time), '{ip_tuple_regex}') THEN trim(attack_subcat)
              ELSE trim(attack_cat)
          END AS attack_cat,

          CASE
              WHEN regexp_full_match(trim(attack_cat), '{ip_tuple_regex}') THEN trim(attack_refrence)
              WHEN regexp_full_match(trim(time), '{ip_tuple_regex}') THEN trim(attack_name)
              ELSE trim(attack_subcat)
          END AS attack_subcat,

          CASE
              WHEN regexp_full_match(trim(attack_cat), '{ip_tuple_regex}') THEN trim(ips)
              WHEN regexp_full_match(trim(time), '{ip_tuple_regex}') THEN trim(attack_refrence)
              ELSE trim(attack_name)
          END AS attack_name,

          CASE
              WHEN regexp_full_match(trim(attack_cat), '{ip_tuple_regex}') THEN trim(time)
              WHEN regexp_full_match(trim(time), '{ip_tuple_regex}') THEN trim(ips)
              ELSE trim(attack_refrence)
          END AS attack_refrence,

          CASE
              WHEN regexp_full_match(trim(attack_cat), '{ip_tuple_regex}') THEN trim(attack_cat)
              WHEN regexp_full_match(trim(time), '{ip_tuple_regex}') THEN trim(time)
              ELSE trim(ips)
          END AS ips

      FROM '{ground_truth}'
      WHERE time != 'Time'
  ) TO '{fixed_ground_truth}' (FORMAT PARQUET)
  """)

  fixed_ground_truth.replace(ground_truth)


PosixPath('../dataset/ground_truth.parquet')

In [11]:
query_df(f"select distinct attack_cat from '{ground_truth}'")


,attack_cat
0,Denial of Service
1,Shellcode
2,Worms
3,Fuzzers
4,Generic
5,Reconnaissance
6,Backdoors
7,Exploits
8,Malware


In [12]:
query_df(f"""
SELECT
    count(*) AS total_rows,
  count(*) - count(DISTINCT row(date, time, attack_cat, attack_subcat, attack_name, attack_refrence, ips)) AS duplicate_rows
FROM '{ground_truth}'
""")


,total_rows,duplicate_rows
0,313900,2292


In [13]:
dedup_ground_truth = dataset_root / "ground_truth_dedup_tmp.parquet"

con.execute(f"""
COPY (
  SELECT DISTINCT *
  FROM '{ground_truth}'
) TO '{dedup_ground_truth}'
  """)
dedup_ground_truth.replace(ground_truth)

PosixPath('../dataset/ground_truth.parquet')

In [14]:
query_df(f"""
SELECT
    count(*) AS total_rows,
  count(*) - count(DISTINCT row(date, time, attack_cat, attack_subcat, attack_name, attack_refrence, ips)) AS duplicate_rows
FROM '{ground_truth}'
""")

,total_rows,duplicate_rows
0,311608,0


In [15]:
query_df(f"select count(*) FROM '{host_logs}' where label=1")

,count_star()
0,1262427


In [16]:
query_df(f"""
SELECT
  count(*) AS total_rows,
  count(DISTINCT row(date, time, pro_id, path, sys_call, event_id, attack_cat, attack_subcat, label)) AS unique_rows,
  count(*) - count(DISTINCT row(date, time, pro_id, path, sys_call, event_id, attack_cat, attack_subcat, label)) AS duplicate_rows
FROM '{host_logs}'
""")


,total_rows,unique_rows,duplicate_rows
0,90054239,89709941,344298


In [17]:
  dedup_host_logs = dataset_root / "host_logs_dedup_tmp.parquet"

  con.execute(f"""
  COPY (
      SELECT DISTINCT *
      FROM '{host_logs}'
  ) TO '{dedup_host_logs}'
  """)

  dedup_host_logs.replace(host_logs)

PosixPath('../dataset/host_logs.parquet')

In [18]:
  query_df(f"""
  SELECT
      count(*) AS total_rows,
      count(DISTINCT row(date, time, pro_id, path, sys_call, event_id, attack_cat, attack_subcat, label)) AS unique_rows,
      count(*) - count(DISTINCT row(date, time, pro_id, path, sys_call, event_id, attack_cat, attack_subcat, label)) AS duplicate_rows
  FROM '{host_logs}'
  """)


,total_rows,unique_rows,duplicate_rows
0,89709941,89709941,0


In [19]:
query_df(f"select * from '{ground_truth}' where attack_cat='Backdoors'")

,date,time,attack_cat,attack_subcat,attack_name,attack_refrence,ips
0,2016-03-14,10:48:00,Backdoors,All Batch,Cisco Network Registrar Default Credentials Ba...,CVE 2011-2024 (http://cve.mitre.org/cgi-bin/cv...,IP 175.45.176.2:0->10.40.85.32:0 175.45.176.2:...
1,2016-03-15,1:40:48,Backdoors,All Batch,BlackEnergy Botnet Command and Control Communi...,http://atlas-public.ec2.arbor.net/docs/BlackEn...,175.45.176.3:28187->10.40.85.32:60357
2,2016-03-11,8:52:48,Backdoors,All Batch,BlackEnergy Botnet Command and Control Communi...,http://atlas-public.ec2.arbor.net/docs/BlackEn...,175.45.176.0:46739->10.40.85.32:59917
3,2016-03-11,8:09:36,Backdoors,All Batch,Cisco Network Registrar Default Credentials Ba...,CVE 2011-2024 (http://cve.mitre.org/cgi-bin/cv...,IP 175.45.176.3:0->10.40.85.32:0 175.45.176.3:...
4,2016-03-11,9:07:12,Backdoors,All Batch,Symantec LiveUpdate Administrator Security Byp...,CVE 2014-1644 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.1:44692->10.40.85.32:80
...,...,...,...,...,...,...,...
1195,2016-03-12,1:26:24,Backdoors,All Batch,Zavio IP Camera Firmware 1.6.03 User Credentia...,CVE 2013-2568 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.2:37712->10.40.85.32:80
1196,2016-03-11,6:43:12,Backdoors,All Batch,Symantec LiveUpdate Administrator Security Byp...,CVE 2014-1644 (http://cve.mitre.org/cgi-bin/cv...,175.45.176.0:32383->10.40.85.32:80
1197,2016-03-11,5:16:48,Backdoors,All Batch,Android AndroidKungFu Malware Command and Cont...,http://about-threats.trendmicro.com/malware.as...,175.45.176.1:51416->10.40.85.32:7500
1198,2016-03-11,6:00:00,Backdoors,All Batch,Backdoor: Windows 7 CMD.EXE Reverse Shell (htt...,Backdoor: Windows 7 CMD.EXE Reverse Shell (htt...,175.45.176.0:13765->10.40.85.32:27469


In [20]:
query_df(f"select distinct attack_cat from '{host_logs}' where date='2016-03-14' and time='8:52:48'")

,attack_cat
0,Backdoors


In [21]:
  window_seconds=60
  query_df(f"""
  WITH attack AS (
      SELECT TIMESTAMP '{attack_date} {attack_time}' AS attack_ts
  )
  SELECT
      h.pro_id,
      h.path,
      count(*) AS event_count,
      sum(h.label) AS anomaly_events
  FROM '{host_logs}' h
  CROSS JOIN attack a
  WHERE h.date + h.time BETWEEN a.attack_ts - INTERVAL {window_seconds} SECOND
                            AND a.attack_ts + INTERVAL {window_seconds} SECOND
  GROUP BY h.pro_id, h.path
  HAVING sum(h.label) > 0
  ORDER BY anomaly_events DESC, event_count DESC
  """)


NameError: name 'attack_date' is not defined

In [22]:
query_df(f"select * from '{host_logs}' limit 10")

,date,time,pro_id,path,sys_call,event_id,attack_cat,attack_subcat,label
0,2016-03-11,03:45:00,1804,/bin/dbus-daemon,256,1022327,normal,normal,0
1,2016-03-11,03:51:40,4493,/usr/sbin/apache2,78,1118242,normal,normal,0
2,2016-03-11,03:47:40,4519,/usr/sbin/apache2,78,1057986,normal,normal,0
3,2016-03-11,03:44:47,2110,/usr/bin/compiz,192,996570,normal,normal,0
4,2016-03-11,03:51:43,2110,/usr/bin/compiz,265,1119139,normal,normal,0
5,2016-03-11,03:51:43,4493,/usr/sbin/apache2,256,1119126,normal,normal,0
6,2016-03-11,03:51:45,2110,/usr/bin/compiz,102,1120139,normal,normal,0
7,2016-03-11,03:51:46,1081,/usr/bin/Xorg,142,1120440,normal,normal,0
8,2016-03-11,03:47:46,2319,/usr/bin/unity-scope-loader,292,1059832,normal,normal,0
9,2016-03-11,03:47:17,4519,/usr/lib/libreoffice/program/soffice.bin,78,1049709,Exploits,Office Document Batch,1


In [23]:
  train_path = dataset_root / "host_logs_train.parquet"
  test_path = dataset_root / "host_logs_test.parquet"

  con.execute(f"""
  COPY (
      SELECT *
      FROM '{host_logs}'
      WHERE date <= DATE '2016-03-15'
  ) TO '{train_path}'
  """)


  con.execute(f"""
  COPY (
      SELECT *
      FROM '{host_logs}'
      WHERE date = DATE '2016-03-16'
  ) TO '{test_path}'
  """)


In [24]:
  query_df(f"""
  SELECT
      label,
      count(*) AS row_count,
      round(100.0 * count(*) / sum(count(*)) OVER (), 4) AS percentage
  FROM '{train_path}'
  GROUP BY label
  ORDER BY label
  """)


,label,row_count,percentage
0,0,79042624,98.631
1,1,1097092,1.369


In [25]:
  query_df(f"""
  SELECT
      label,
      count(*) AS row_count,
      round(100.0 * count(*) / sum(count(*)) OVER (), 4) AS percentage
  FROM '{test_path}'
  GROUP BY label
  ORDER BY label
  """)


,label,row_count,percentage
0,0,9409821,98.3239
1,1,160404,1.6761


In [26]:
query_df(f"select distinct attack_cat from '{train_path}' where label=1")

,attack_cat
0,Denial of Service
1,Shellcode
2,Worms
3,Exploits
4,Backdoors
5,Reconnaissance
6,Generic


In [27]:
query_df(f"select distinct attack_cat from '{test_path}' where label=1")

,attack_cat
0,Generic
1,Reconnaissance
2,Denial of Service
3,Shellcode
4,Worms
5,Exploits
6,Backdoors


# Feature Engineering

Dataset cleaning, sequence-step construction, syscall lookup mapping, model-feature creation, and train/test parquet preparation.


In [28]:
host_sequence_path = dataset_root / "host_logs_sequence.parquet"

con.execute(f"""
COPY (
  SELECT
      *,
      row_number() OVER (
          ORDER BY date, time, event_id
      ) AS sequence_step
  FROM '{host_logs}'
  ORDER BY date, time, event_id
) TO '{host_sequence_path}' (FORMAT PARQUET)
""")
host_sequence_path.replace(host_logs)

PosixPath('../dataset/host_logs.parquet')

In [29]:
query_df(f"select * from '{host_logs}' limit 10")

,date,time,pro_id,path,sys_call,event_id,attack_cat,attack_subcat,label,sequence_step
0,2016-03-11,02:45:01,1951,/usr/lib/i386-linux-gnu/indicator-datetime/ind...,168,45350,normal,normal,0,1
1,2016-03-11,02:45:01,1966,/usr/lib/i386-linux-gnu/indicator-datetime/ind...,168,45351,normal,normal,0,2
2,2016-03-11,02:45:01,1885,/usr/lib/unity/unity-panel-service,168,45353,normal,normal,0,3
3,2016-03-11,02:45:01,1830,/sbin/upstart-dbus-bridge,142,45354,normal,normal,0,4
4,2016-03-11,02:45:01,1872,/usr/lib/unity/unity-panel-service,168,45355,normal,normal,0,5
5,2016-03-11,02:45:01,2114,/usr/bin/compiz,168,45357,normal,normal,0,6
6,2016-03-11,02:45:06,1804,/bin/dbus-daemon,256,45352,normal,normal,0,7
7,2016-03-11,02:45:06,2834,/usr/bin/update-notifier,142,45360,normal,normal,0,8
8,2016-03-11,02:45:06,2133,/usr/lib/i386-linux-gnu/gconf/gconfd-2,168,45372,normal,normal,0,9
9,2016-03-11,02:45:11,3989,/sbin/auditd,256,45374,normal,normal,0,10


In [30]:
query_df(f"""
SELECT *
FROM '{host_logs}'
WHERE date IS NULL
 OR time IS NULL
 OR pro_id IS NULL
 OR path IS NULL
 OR sys_call IS NULL
 OR event_id IS NULL
 OR attack_cat IS NULL
 OR attack_subcat IS NULL
 OR label IS NULL
LIMIT 100
""")


,date,time,pro_id,path,sys_call,event_id,attack_cat,attack_subcat,label,sequence_step


In [31]:
query_df(f"""
    SELECT *
    FROM '{host_logs}'
    WHERE date IS NULL
    OR time IS NULL
    OR pro_id IS NULL
    OR path IS NULL
    OR trim(path) = ''
    OR sys_call IS NULL
    OR event_id IS NULL
    OR attack_cat IS NULL
    OR trim(attack_cat) = ''
    OR attack_subcat IS NULL
    OR trim(attack_subcat) = ''
    OR label IS NULL
    LIMIT 100
""")


,date,time,pro_id,path,sys_call,event_id,attack_cat,attack_subcat,label,sequence_step


In [32]:
  clean_host_logs = dataset_root / "host_logs_clean_tmp.parquet"

  con.execute(f"""
  COPY (
      SELECT *
      FROM '{host_logs}'
      WHERE date IS NOT NULL
        AND time IS NOT NULL
        AND pro_id IS NOT NULL
        AND path IS NOT NULL
        AND trim(path) != ''
        AND sys_call IS NOT NULL
        AND event_id IS NOT NULL
        AND attack_cat IS NOT NULL
        AND trim(attack_cat) != ''
        AND attack_subcat IS NOT NULL
        AND trim(attack_subcat) != ''
        AND label IS NOT NULL
  ) TO '{clean_host_logs}' (FORMAT PARQUET)
  """)

  clean_host_logs.replace(host_logs)


PosixPath('../dataset/host_logs.parquet')

In [33]:
  query_df(f"""
  SELECT *
  FROM '{host_logs}'
  WHERE date IS NULL
     OR time IS NULL
     OR pro_id IS NULL
     OR path IS NULL
     OR trim(path) = ''
     OR sys_call IS NULL
     OR event_id IS NULL
     OR attack_cat IS NULL
     OR trim(attack_cat) = ''
     OR attack_subcat IS NULL
     OR trim(attack_subcat) = ''
     OR label IS NULL
  LIMIT 100
  """)


,date,time,pro_id,path,sys_call,event_id,attack_cat,attack_subcat,label,sequence_step


In [34]:
host_features_path = dataset_root / "host_logs_sequence_features.parquet"

con.execute(f"""
COPY (
  SELECT
      *,
      coalesce(
          datediff(
              'second',
              lag(date + time) OVER (
                  ORDER BY sequence_step
              ),
              date + time
          ),
          0
      ) AS delta_seconds
  FROM '{host_logs}'
  ORDER BY sequence_step
) TO '{host_features_path}'
""")
host_features_path.replace(host_logs)

PosixPath('../dataset/host_logs.parquet')

In [35]:
syscall_table = dataset_root / "syscall_32.tbl"
syscall_lookup = dataset_root / "syscall_32.parquet"

con.execute(f"""
COPY (
    SELECT
        column0 AS sys_call,
        column1 AS abi,
        column2 AS syscall_name
    FROM read_csv(
        '{syscall_table}',
        delim='\t',
        header=false,
        comment='#',
        null_padding=true,
        ignore_errors=true
    )
) TO '{syscall_lookup}'
""")

In [36]:
host_model_features_path = dataset_root / "host_logs_model_features.parquet"

con.execute(f"""
COPY (
    WITH ordered_logs AS (
        SELECT
            h.date,
            h.time,
            h.pro_id,
            h.path,
            h.sys_call,
            h.event_id,
            h.attack_cat,
            h.attack_subcat,
            h.label,
            row_number() OVER (
                ORDER BY h.date, h.time, h.event_id
            ) AS sequence_step,
            coalesce(
                datediff(
                    'second',
                    lag(h.date + h.time) OVER (
                        ORDER BY h.date, h.time, h.event_id
                    ),
                    h.date + h.time
                ),
                0
            ) AS delta_seconds
        FROM '{host_logs}' h
    ),
    syscall_vocab AS (
        SELECT
            sys_call,
            row_number() OVER (ORDER BY sys_call) AS sys_call_token
        FROM (
            SELECT DISTINCT sys_call
            FROM ordered_logs
        )
    )
    SELECT
        h.date,
        h.time,
        h.event_id,
        h.sequence_step,
        h.sys_call,
        v.sys_call_token,
        s.syscall_name,
        h.delta_seconds,
        least(
            log(1 + greatest(h.delta_seconds, 0)),
            10.0
        ) AS delta_time_log,
        h.label,

        -- Attack details are kept as metadata for analysis and XAI reporting.
        -- They must not be used as LSTM input features.
        h.attack_cat,
        h.attack_subcat
    FROM ordered_logs h
    LEFT JOIN syscall_vocab v
        ON h.sys_call = v.sys_call
    LEFT JOIN '{syscall_lookup}' s
        ON h.sys_call = s.sys_call
    ORDER BY h.sequence_step
) TO '{host_model_features_path}' (FORMAT PARQUET)
""")

host_model_features_path.replace(host_logs)


PosixPath('../dataset/host_logs.parquet')

In [37]:
query_df(f"select * from '{host_logs}' limit 10")

,date,time,event_id,sequence_step,sys_call,sys_call_token,syscall_name,delta_seconds,delta_time_log,label,attack_cat,attack_subcat
0,2016-03-11,02:45:01,45350,1,168,59,poll,0,0.000000,0,normal,normal
1,2016-03-11,02:45:01,45351,2,168,59,poll,0,0.000000,0,normal,normal
2,2016-03-11,02:45:01,45353,3,168,59,poll,0,0.000000,0,normal,normal
3,2016-03-11,02:45:01,45354,4,142,52,_newselect,0,0.000000,0,normal,normal
4,2016-03-11,02:45:01,45355,5,168,59,poll,0,0.000000,0,normal,normal
5,2016-03-11,02:45:01,45357,6,168,59,poll,0,0.000000,0,normal,normal
6,2016-03-11,02:45:06,45352,7,256,97,epoll_wait,5,0.778151,0,normal,normal
7,2016-03-11,02:45:06,45360,8,142,52,_newselect,0,0.000000,0,normal,normal
8,2016-03-11,02:45:06,45372,9,168,59,poll,0,0.000000,0,normal,normal
9,2016-03-11,02:45:11,45374,10,256,97,epoll_wait,5,0.778151,0,normal,normal


In [38]:
train_features_path = dataset_root / "host_log_train.parquet"
test_features_path = dataset_root / "host_log_test.parquet"

con.execute(f"""
COPY (
    SELECT *
    FROM '{host_logs}'
    WHERE date <= DATE '2016-03-15'
    ORDER BY sequence_step
) TO '{train_features_path}' (FORMAT PARQUET)
""")

con.execute(f"""
COPY (
    SELECT *
    FROM '{host_logs}'
    WHERE date = DATE '2016-03-16'
    ORDER BY sequence_step
) TO '{test_features_path}' (FORMAT PARQUET)
""")

print(f"train_features_path: {train_features_path}")
print(f"test_features_path: {test_features_path}")


train_features_path: ../dataset/host_log_train.parquet
test_features_path: ../dataset/host_log_test.parquet


In [39]:
query_df(f"describe '{host_logs}'")

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,time,TIME,YES,None,None,None
2,event_id,BIGINT,YES,None,None,None
3,sequence_step,BIGINT,YES,None,None,None
4,sys_call,BIGINT,YES,None,None,None
5,sys_call_token,BIGINT,YES,None,None,None
6,syscall_name,VARCHAR,YES,None,None,None
7,delta_seconds,BIGINT,YES,None,None,None
8,delta_time_log,DOUBLE,YES,None,None,None
9,label,BIGINT,YES,None,None,None


# Model Training and Testing

LSTM window construction, model training, model testing, and XAI comparison outputs for user-centric evaluation.


In [ ]:
import numpy as np

WINDOW_SIZE = 64
STRIDE = 32


def summarize_window_attack_metadata(window):
    attack_rows = window[window["label"] == 1]
    if attack_rows.empty:
        return {
            "window_label": 0,
            "attack_cat": "normal",
            "attack_subcat": "normal",
        }

    attack_cat = ", ".join(
        sorted(attack_rows["attack_cat"].dropna().astype(str).unique())
    )
    attack_subcat = ", ".join(
        sorted(attack_rows["attack_subcat"].dropna().astype(str).unique())
    )

    return {
        "window_label": 1,
        "attack_cat": attack_cat if attack_cat else "unknown",
        "attack_subcat": attack_subcat if attack_subcat else "unknown",
    }


def build_lstm_windows(feature_path, window_size=64, stride=32):
    df = query_df(f"""
    SELECT
        sequence_step,
        sys_call_token,
        delta_time_log,
        label,
        attack_cat,
        attack_subcat
    FROM '{feature_path}'
    ORDER BY sequence_step
    """)

    syscalls = df["sys_call_token"].to_numpy(dtype=np.int32)
    delta_time = df["delta_time_log"].to_numpy(dtype=np.float32)
    labels = df["label"].to_numpy(dtype=np.int32)

    x_syscall = []
    x_delta = []
    y = []
    window_metadata = []

    for start in range(0, len(df) - window_size + 1, stride):
        end = start + window_size
        window = df.iloc[start:end]
        metadata = summarize_window_attack_metadata(window)

        x_syscall.append(syscalls[start:end])
        x_delta.append(delta_time[start:end])
        y.append(labels[start:end].max())
        window_metadata.append({
            "window_start_sequence_step": int(window["sequence_step"].iloc[0]),
            "window_end_sequence_step": int(window["sequence_step"].iloc[-1]),
            **metadata,
        })

    x_syscall = np.array(x_syscall, dtype=np.int32)
    x_delta = np.array(x_delta, dtype=np.float32).reshape(-1, window_size, 1)
    y = np.array(y, dtype=np.int32)
    window_metadata = pd.DataFrame(window_metadata)

    return x_syscall, x_delta, y, window_metadata


X_train_syscall, X_train_delta, y_train, train_window_metadata = build_lstm_windows(
    train_features_path,
    WINDOW_SIZE,
    STRIDE,
)

X_test_syscall, X_test_delta, y_test, test_window_metadata = build_lstm_windows(
    test_features_path,
    WINDOW_SIZE,
    STRIDE,
)

print("X_train_syscall:", X_train_syscall.shape)
print("X_train_delta:", X_train_delta.shape)
print("y_train:", y_train.shape)
print("X_test_syscall:", X_test_syscall.shape)
print("X_test_delta:", X_test_delta.shape)
print("y_test:", y_test.shape)

print("Train positive windows:", int(y_train.sum()))
print("Train negative windows:", int(len(y_train) - y_train.sum()))
print("Test positive windows:", int(y_test.sum()))
print("Test negative windows:", int(len(y_test) - y_test.sum()))

print("Train attack metadata sample")
display(train_window_metadata[train_window_metadata["window_label"] == 1].head())

print("Test attack metadata sample")
display(test_window_metadata[test_window_metadata["window_label"] == 1].head())


In [ ]:
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Token 0 is reserved for unknown/padding.
syscall_vocab_size = int(max(X_train_syscall.max(), X_test_syscall.max())) + 1
print("syscall_vocab_size:", syscall_vocab_size)

# Handle class imbalance at the window level.
negative_count = int(len(y_train) - y_train.sum())
positive_count = int(y_train.sum())

class_weight = {
    0: 1.0,
    1: negative_count / positive_count,
}

print("class_weight:", class_weight)

syscall_input = tf.keras.Input(
    shape=(WINDOW_SIZE,),
    name="sys_call_token",
)

delta_input = tf.keras.Input(
    shape=(WINDOW_SIZE, 1),
    name="delta_time_log",
)

syscall_embedding = tf.keras.layers.Embedding(
    input_dim=syscall_vocab_size,
    output_dim=32,
    name="syscall_embedding",
)(syscall_input)

x = tf.keras.layers.Concatenate(name="event_features")([
    syscall_embedding,
    delta_input,
])

x = tf.keras.layers.LSTM(
    64,
    dropout=0.2,
    recurrent_dropout=0.0,
    name="lstm_layer",
)(x)

x = tf.keras.layers.Dense(32, activation="relu", name="dense_layer")(x)
x = tf.keras.layers.Dropout(0.3, name="dropout_layer")(x)

output = tf.keras.layers.Dense(
    1,
    activation="sigmoid",
    name="anomaly_score",
)(x)

model = tf.keras.Model(
    inputs=[syscall_input, delta_input],
    outputs=output,
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(curve="ROC", name="roc_auc"),
        tf.keras.metrics.AUC(curve="PR", name="pr_auc"),
    ],
)

model.summary()

history = model.fit(
    {
        "sys_call_token": X_train_syscall,
        "delta_time_log": X_train_delta,
    },
    y_train,
    epochs=5,
    batch_size=256,
    class_weight=class_weight,
    verbose=1,
)

test_results = model.evaluate(
    {
        "sys_call_token": X_test_syscall,
        "delta_time_log": X_test_delta,
    },
    y_test,
    batch_size=256,
    verbose=1,
)

print(dict(zip(model.metrics_names, test_results)))

y_test_score = model.predict(
    {
        "sys_call_token": X_test_syscall,
        "delta_time_log": X_test_delta,
    },
    batch_size=256,
).ravel()

threshold = 0.5
y_test_pred = (y_test_score >= threshold).astype(int)

print(confusion_matrix(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred, digits=4))







In [ ]:
# Simple SHAP explanation for one high-score test window.
# Output is structured for the questionnaire: input, anomaly reason, model output, and SHAP explanation.
try:
    import shap
except ImportError:
    import sys
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap"])
    import shap


SYSCALL_BEHAVIOR_MAP = {
    "read": "file read activity",
    "write": "file write activity",
    "open": "file access",
    "close": "file access",
    "stat": "file metadata access",
    "fstat": "file metadata access",
    "lstat": "file metadata access",
    "access": "permission check",
    "chmod": "permission change",
    "chown": "ownership change",
    "execve": "process execution",
    "fork": "process creation",
    "clone": "process or thread creation",
    "socketcall": "network communication",
    "connect": "network connection",
    "accept": "network connection",
    "send": "network communication",
    "recv": "network communication",
    "mmap": "memory mapping",
    "munmap": "memory management",
    "mprotect": "memory protection",
    "brk": "memory allocation",
    "poll": "event polling",
    "select": "event polling",
    "ioctl": "device or terminal control",
}


def behavior_label(syscall_name):
    return SYSCALL_BEHAVIOR_MAP.get(str(syscall_name), "other system activity")


def flatten_lstm_inputs(x_syscall, x_delta):
    return np.concatenate(
        [
            x_syscall.astype(np.float32),
            x_delta.reshape(x_delta.shape[0], -1).astype(np.float32),
        ],
        axis=1,
    )


def unflatten_lstm_inputs(x_flat):
    syscall_part = np.rint(x_flat[:, :WINDOW_SIZE]).astype(np.int32)
    syscall_part = np.clip(syscall_part, 0, syscall_vocab_size - 1)
    delta_part = x_flat[:, WINDOW_SIZE:].reshape(-1, WINDOW_SIZE, 1).astype(np.float32)
    return {
        "sys_call_token": syscall_part,
        "delta_time_log": delta_part,
    }


def predict_flat_lstm(x_flat):
    return model.predict(unflatten_lstm_inputs(x_flat), verbose=0).ravel()


# For the questionnaire, use a captured anomaly: ground truth anomaly and model prediction anomaly.
true_positive_indices = np.where((y_test == 1) & (y_test_score >= threshold))[0]

if len(true_positive_indices) > 0:
    sample_index = int(true_positive_indices[np.argmax(y_test_score[true_positive_indices])])
    sample_selection_note = "Selected highest-score true positive: labelled anomaly and predicted anomaly."
else:
    true_anomaly_indices = np.where(y_test == 1)[0]
    sample_index = int(true_anomaly_indices[np.argmax(y_test_score[true_anomaly_indices])])
    sample_selection_note = "No true positive found at current threshold. Selected highest-score labelled anomaly."

background_size = min(30, len(X_train_syscall))
background_indices = np.random.choice(len(X_train_syscall), size=background_size, replace=False)

background = flatten_lstm_inputs(
    X_train_syscall[background_indices],
    X_train_delta[background_indices],
)

sample_flat = flatten_lstm_inputs(
    X_test_syscall[sample_index:sample_index + 1],
    X_test_delta[sample_index:sample_index + 1],
)

feature_names = (
    [f"syscall_t{i}" for i in range(WINDOW_SIZE)]
    + [f"delta_t{i}" for i in range(WINDOW_SIZE)]
)

explainer = shap.KernelExplainer(predict_flat_lstm, background)
shap_values = explainer.shap_values(sample_flat, nsamples=100)
shap_values = np.array(shap_values).reshape(-1)

shap_df = pd.DataFrame({
    "feature": feature_names,
    "value": sample_flat.reshape(-1),
    "shap_value": shap_values,
})
shap_df["abs_shap_value"] = shap_df["shap_value"].abs()

# Lookup table for converting tokens back to 32-bit Linux syscall names.
token_lookup = query_df(f"""
SELECT DISTINCT
    sys_call_token,
    sys_call,
    syscall_name
FROM '{train_features_path}'
ORDER BY sys_call_token
""")

# Full 64-event input window shown in readable form.
sample_tokens = X_test_syscall[sample_index]
input_window = pd.DataFrame({
    "timestep": range(WINDOW_SIZE),
    "sys_call_token": sample_tokens,
})
input_window = input_window.merge(token_lookup, on="sys_call_token", how="left")
input_window["behavior"] = input_window["syscall_name"].apply(behavior_label)
input_window = input_window[
    ["timestep", "sys_call", "syscall_name", "behavior"]
].rename(columns={
    "sys_call": "raw_syscall_id",
    "syscall_name": "linux_32bit_syscall_name",
})

# SHAP contribution table for syscall positions.
syscall_shap = shap_df[shap_df["feature"].str.startswith("syscall_t")].copy()
syscall_shap["timestep"] = syscall_shap["feature"].str.extract(r"syscall_t(\d+)").astype(int)
syscall_shap["sys_call_token"] = syscall_shap["value"].astype(int)
syscall_shap = syscall_shap.merge(token_lookup, on="sys_call_token", how="left")
syscall_shap["behavior"] = syscall_shap["syscall_name"].apply(behavior_label)
syscall_shap["direction"] = np.where(
    syscall_shap["shap_value"] >= 0,
    "pushes toward anomaly",
    "pushes toward normal",
)

behavior_summary = (
    syscall_shap
    .groupby("behavior", as_index=False)["shap_value"]
    .sum()
    .assign(abs_shap_value=lambda df: df["shap_value"].abs())
    .sort_values("abs_shap_value", ascending=False)
)

positive_behaviors = (
    behavior_summary[behavior_summary["shap_value"] > 0]
    .head(3)["behavior"]
    .tolist()
)
behavior_text = ", ".join(positive_behaviors) if positive_behaviors else "low-level system activity"

if "test_window_metadata" in globals():
    metadata = test_window_metadata.iloc[sample_index]
    attack_cat = metadata["attack_cat"]
    attack_subcat = metadata["attack_subcat"]
    window_start = metadata["window_start_sequence_step"]
    window_end = metadata["window_end_sequence_step"]
else:
    attack_cat = "unknown"
    attack_subcat = "unknown"
    window_start = "unknown"
    window_end = "unknown"

# 1. Input shown to participant.
input_summary = pd.DataFrame([
    {"field": "Sample type", "value": f"{WINDOW_SIZE}-event host syscall sequence"},
    {"field": "Sample selection", "value": sample_selection_note},
    {"field": "Syscall mapping", "value": "Linux 32-bit syscall table"},
    {"field": "Window start sequence step", "value": window_start},
    {"field": "Window end sequence step", "value": window_end},
    {"field": "Dataset attack category", "value": attack_cat},
    {"field": "Dataset attack subcategory", "value": attack_subcat},
])

print("1. Input shown to participant")
display(input_summary)

print("Readable input window, first 20 syscall events")
display(input_window.head(20))

# 2. Why this sample is considered anomalous.
anomaly_reason = pd.DataFrame([
    {
        "section": "Observed suspicious behaviour",
        "explanation": f"This syscall window contains behaviour that the model associated with anomaly, mainly: {behavior_text}.",
    },
    {
        "section": "Security interpretation",
        "explanation": "In host monitoring, a short sequence containing these behaviours can be suspicious because exploited applications may execute code, interact with files, change memory/process state, or communicate with other systems.",
    },
    {
        "section": "Dataset ground truth",
        "explanation": f"The dataset labels this window as {attack_cat} / {attack_subcat}.",
    },
])

print("2. Why this sample is considered anomalous")
display(anomaly_reason)

# 3. Model output.
model_output = pd.DataFrame([
    {"field": "Model", "value": "LSTM sequence classifier"},
    {"field": "Prediction", "value": "anomaly" if y_test_score[sample_index] >= threshold else "normal"},
    {"field": "Anomaly score", "value": round(float(y_test_score[sample_index]), 4)},
    {"field": "Decision threshold", "value": threshold},
    {"field": "True label", "value": int(y_test[sample_index])},
])

print("3. Model output")
display(model_output)

# 4. SHAP explanation.
print("4. SHAP explanation: top syscall contributions")
display(
    syscall_shap[
        ["timestep", "sys_call", "syscall_name", "behavior", "shap_value"]
    ]
    .rename(columns={
        "sys_call": "raw_syscall_id",
        "syscall_name": "linux_32bit_syscall_name",
        "shap_value": "contribution",
    })
    .sort_values("contribution", key=lambda x: x.abs(), ascending=False)
    .head(10)
)

print("SHAP behavior contribution summary")
display(
    behavior_summary[
        ["behavior", "shap_value", "abs_shap_value"]
    ]
    .rename(columns={
        "shap_value": "contribution",
        "abs_shap_value": "absolute_contribution",
    })
    .head(10)
)


In [ ]:
# Simple LIME explanation for the same questionnaire sample.
# LIME is used here as a local surrogate approximation of the LSTM prediction.
try:
    from lime.lime_tabular import LimeTabularExplainer
except ImportError:
    import sys
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "lime"])
    from lime.lime_tabular import LimeTabularExplainer


# If the SHAP cell was not run first, define the minimum required context here.
if "sample_index" not in globals():
    true_positive_indices = np.where((y_test == 1) & (y_test_score >= threshold))[0]
    if len(true_positive_indices) > 0:
        sample_index = int(true_positive_indices[np.argmax(y_test_score[true_positive_indices])])
    else:
        true_anomaly_indices = np.where(y_test == 1)[0]
        sample_index = int(true_anomaly_indices[np.argmax(y_test_score[true_anomaly_indices])])

if "feature_names" not in globals():
    feature_names = (
        [f"syscall_t{i}" for i in range(WINDOW_SIZE)]
        + [f"delta_t{i}" for i in range(WINDOW_SIZE)]
    )

if "background" not in globals():
    background_size = min(30, len(X_train_syscall))
    background_indices = np.random.choice(len(X_train_syscall), size=background_size, replace=False)
    background = flatten_lstm_inputs(
        X_train_syscall[background_indices],
        X_train_delta[background_indices],
    )

if "sample_flat" not in globals():
    sample_flat = flatten_lstm_inputs(
        X_test_syscall[sample_index:sample_index + 1],
        X_test_delta[sample_index:sample_index + 1],
    )

if "token_lookup" not in globals():
    token_lookup = query_df(f"""
    SELECT DISTINCT
        sys_call_token,
        sys_call,
        syscall_name
    FROM '{train_features_path}'
    ORDER BY sys_call_token
    """)

lime_explainer = LimeTabularExplainer(
    training_data=background,
    feature_names=feature_names,
    mode="regression",
    discretize_continuous=False,
    random_state=42,
)

lime_exp = lime_explainer.explain_instance(
    sample_flat[0],
    predict_flat_lstm,
    num_features=10,
)

lime_raw = pd.DataFrame(
    lime_exp.as_list(),
    columns=["local_condition", "lime_weight"],
)
lime_raw["feature"] = lime_raw["local_condition"].str.extract(r"(syscall_t\d+|delta_t\d+)")
lime_raw["feature_type"] = np.where(
    lime_raw["feature"].str.startswith("syscall_t", na=False),
    "syscall",
    "timing",
)

lime_syscall = lime_raw[lime_raw["feature_type"] == "syscall"].copy()
lime_syscall["timestep"] = lime_syscall["feature"].str.extract(r"syscall_t(\d+)").astype(int)
lime_syscall["sys_call_token"] = sample_flat[0, lime_syscall["timestep"].to_numpy()].astype(int)

lime_syscall = lime_syscall.merge(token_lookup, on="sys_call_token", how="left")
lime_syscall["behavior"] = lime_syscall["syscall_name"].apply(behavior_label)

lime_timing = lime_raw[lime_raw["feature_type"] == "timing"].copy()
lime_timing["timestep"] = lime_timing["feature"].str.extract(r"delta_t(\d+)").astype(float).astype("Int64")
lime_timing["sys_call"] = np.nan
lime_timing["syscall_name"] = "timing gap"
lime_timing["behavior"] = "timing pattern"

lime_display = pd.concat(
    [
        lime_syscall[
            ["local_condition", "timestep", "sys_call", "syscall_name", "behavior", "lime_weight"]
        ],
        lime_timing[
            ["local_condition", "timestep", "sys_call", "syscall_name", "behavior", "lime_weight"]
        ],
    ],
    ignore_index=True,
)

lime_display = lime_display.rename(columns={
    "sys_call": "raw_syscall_id",
    "syscall_name": "linux_32bit_syscall_name",
    "lime_weight": "effect_on_explanation",
})

lime_display["effect_direction"] = np.where(
    lime_display["effect_on_explanation"] >= 0,
    "increased anomaly score",
    "decreased anomaly score",
)

lime_display = lime_display.sort_values("timestep")

lime_summary = pd.DataFrame([
    {
        "field": "Explanation method",
        "value": "LIME",
    },
    {
        "field": "What it explains",
        "value": "A simple local surrogate approximation of the LSTM prediction for this one window.",
    },
    {
        "field": "Prediction being approximated",
        "value": "anomaly" if y_test_score[sample_index] >= threshold else "normal",
    },
    {
        "field": "Anomaly score",
        "value": round(float(y_test_score[sample_index]), 4),
    },
])

print("LIME explanation: local surrogate approximation")
display(lime_summary)

print("Top LIME local conditions")
display(lime_display.head(10))


In [ ]:
# Simple counterfactual explanation for the same questionnaire sample.
# Counterfactual question: what small changes would make the model score drop below the anomaly threshold?

if "sample_index" not in globals():
    true_positive_indices = np.where((y_test == 1) & (y_test_score >= threshold))[0]
    if len(true_positive_indices) > 0:
        sample_index = int(true_positive_indices[np.argmax(y_test_score[true_positive_indices])])
    else:
        true_anomaly_indices = np.where(y_test == 1)[0]
        sample_index = int(true_anomaly_indices[np.argmax(y_test_score[true_anomaly_indices])])

if "token_lookup" not in globals():
    token_lookup = query_df(f"""
    SELECT DISTINCT
        sys_call_token,
        sys_call,
        syscall_name
    FROM '{train_features_path}'
    ORDER BY sys_call_token
    """)

if "behavior_label" not in globals():
    def behavior_label(syscall_name):
        return str(syscall_name)

# Normal baseline values are estimated from normal training windows only.
normal_train_syscalls = X_train_syscall[y_train == 0].reshape(-1)
normal_train_delta = X_train_delta[y_train == 0].reshape(-1)

baseline_syscall_token = int(pd.Series(normal_train_syscalls).mode().iloc[0])
baseline_delta_time = float(np.median(normal_train_delta))

original_syscalls = X_test_syscall[sample_index:sample_index + 1].copy()
original_delta = X_test_delta[sample_index:sample_index + 1].copy()

original_score = float(model.predict(
    {
        "sys_call_token": original_syscalls,
        "delta_time_log": original_delta,
    },
    verbose=0,
).ravel()[0])

# Rank timesteps by how much the score drops when each timestep is individually replaced by the normal baseline.
occlusion_rows = []
for timestep in range(WINDOW_SIZE):
    changed_syscalls = original_syscalls.copy()
    changed_delta = original_delta.copy()
    original_token = int(changed_syscalls[0, timestep])
    original_delta_value = float(changed_delta[0, timestep, 0])

    changed_syscalls[0, timestep] = baseline_syscall_token
    changed_delta[0, timestep, 0] = baseline_delta_time

    changed_score = float(model.predict(
        {
            "sys_call_token": changed_syscalls,
            "delta_time_log": changed_delta,
        },
        verbose=0,
    ).ravel()[0])

    occlusion_rows.append({
        "timestep": timestep,
        "original_sys_call_token": original_token,
        "score_after_single_change": changed_score,
        "score_drop": original_score - changed_score,
        "original_delta_time_log": original_delta_value,
    })

occlusion_df = pd.DataFrame(occlusion_rows).sort_values("score_drop", ascending=False)

# Greedily apply the most score-reducing changes until the model is no longer anomalous.
counterfactual_syscalls = original_syscalls.copy()
counterfactual_delta = original_delta.copy()
applied_changes = []

for row in occlusion_df.itertuples(index=False):
    if row.score_drop <= 0:
        continue

    timestep = int(row.timestep)
    original_token = int(counterfactual_syscalls[0, timestep])
    original_delta_value = float(counterfactual_delta[0, timestep, 0])

    counterfactual_syscalls[0, timestep] = baseline_syscall_token
    counterfactual_delta[0, timestep, 0] = baseline_delta_time

    new_score = float(model.predict(
        {
            "sys_call_token": counterfactual_syscalls,
            "delta_time_log": counterfactual_delta,
        },
        verbose=0,
    ).ravel()[0])

    applied_changes.append({
        "timestep": timestep,
        "original_sys_call_token": original_token,
        "replacement_sys_call_token": baseline_syscall_token,
        "original_delta_time_log": original_delta_value,
        "replacement_delta_time_log": baseline_delta_time,
        "score_after_change": new_score,
    })

    if new_score < threshold:
        break

counterfactual_score = float(model.predict(
    {
        "sys_call_token": counterfactual_syscalls,
        "delta_time_log": counterfactual_delta,
    },
    verbose=0,
).ravel()[0])

counterfactual_summary = pd.DataFrame([
    {"field": "Explanation method", "value": "Counterfactual"},
    {"field": "Question answered", "value": "What small input changes would make the model stop predicting anomaly?"},
    {"field": "Original anomaly score", "value": round(original_score, 4)},
    {"field": "Counterfactual anomaly score", "value": round(counterfactual_score, 4)},
    {"field": "Decision threshold", "value": threshold},
    {"field": "Number of changed timesteps", "value": len(applied_changes)},
    {"field": "Counterfactual result", "value": "below threshold" if counterfactual_score < threshold else "still above threshold"},
])

changes_df = pd.DataFrame(applied_changes)

if not changes_df.empty:
    original_lookup = token_lookup.rename(columns={
        "sys_call_token": "original_sys_call_token",
        "sys_call": "original_raw_syscall_id",
        "syscall_name": "original_syscall_name",
    })
    replacement_lookup = token_lookup.rename(columns={
        "sys_call_token": "replacement_sys_call_token",
        "sys_call": "replacement_raw_syscall_id",
        "syscall_name": "replacement_syscall_name",
    })

    changes_df = changes_df.merge(original_lookup, on="original_sys_call_token", how="left")
    changes_df = changes_df.merge(replacement_lookup, on="replacement_sys_call_token", how="left")
    changes_df["original_behavior"] = changes_df["original_syscall_name"].apply(behavior_label)
    changes_df["replacement_behavior"] = changes_df["replacement_syscall_name"].apply(behavior_label)

    changes_display = changes_df[
        [
            "timestep",
            "original_raw_syscall_id",
            "original_syscall_name",
            "original_behavior",
            "replacement_raw_syscall_id",
            "replacement_syscall_name",
            "replacement_behavior",
            "score_after_change",
        ]
    ].rename(columns={
        "original_raw_syscall_id": "original_raw_syscall_id",
        "original_syscall_name": "original_32bit_syscall_name",
        "replacement_raw_syscall_id": "replacement_raw_syscall_id",
        "replacement_syscall_name": "replacement_32bit_syscall_name",
    })
else:
    changes_display = pd.DataFrame(columns=[
        "timestep",
        "original_raw_syscall_id",
        "original_32bit_syscall_name",
        "original_behavior",
        "replacement_raw_syscall_id",
        "replacement_32bit_syscall_name",
        "replacement_behavior",
        "score_after_change",
    ])

print("Counterfactual explanation")
display(counterfactual_summary)

print("Counterfactual changes applied")
display(changes_display)

print("Interpretation")
display(pd.DataFrame([
    {
        "explanation": "This is a model-based what-if explanation. It does not claim these changes are realistic edits to a real attack. It shows which syscall positions, if replaced by normal baseline activity, would most reduce the model's anomaly score."
    }
]))


In [ ]:
query_df(f"select distinct path from '{host_logs}'")